In [0]:
from delta.tables import DeltaTable
import pyspark.sql.functions as F

# Catalog and table names

catalog_name = "ecommerce"

source_table_name = "gld_fact_order_items"
table_name = "gld_fact_daily_orders_summary"

# Recalculate the last N days every run
days_cutoff = 30

In [0]:
# Find latest transaction date in fact table

max_date = (
    spark.sql(
        f"""
        SELECT MAX(transaction_date) AS max_date
        FROM {catalog_name}.gold.{source_table_name}
        """
    )
    .collect()[0]["max_date"]
)

print(max_date)

# First run = load everything
# Later runs = only recalculate recent data

if spark.catalog.tableExists(f"{catalog_name}.gold.{table_name}"):

    where_clause = f"""
    transaction_date >=
    date_sub(date('{max_date}'), {days_cutoff})
    """

else:

    where_clause = "1 = 1"

2025-08-31


In [0]:
summary_query = f"""
SELECT

    date_id,

    unit_price_currency AS currency,

    SUM(quantity) AS total_quantity,

    SUM(gross_amount) AS total_gross_amount,

    SUM(discount_amount) AS total_discount_amount,

    SUM(tax_amount) AS total_tax_amount,

    SUM(net_amount) AS total_amount

FROM {catalog_name}.gold.{source_table_name}

WHERE {where_clause}

GROUP BY

    date_id,
    currency

ORDER BY date_id DESC
"""

summary_df = spark.sql(summary_query)

# display(summary_df)

date_id,currency,total_quantity,total_gross_amount,total_discount_amount,total_tax_amount,total_amount
20250831,AUD,145,36808.0,3189,3507,37126.0
20250831,INR,1547,2.0840535E7,1643337,2081652,2.127885E7
20250831,GBP,315,61483.0,4958,6554,63079.0
20250831,CAD,135,26237.0,2099,3149,27287.0
20250831,USD,332,44975.0,3390,4920,46505.0
20250831,AED,218,116752.0,10011,11713,118454.0
20250831,SGD,180,47111.0,4198,5014,47927.0
20250830,INR,2759,3.7118796E7,3480250,3803104,3.744165E7
20250830,USD,567,78650.0,7069,7940,79521.0
20250830,AUD,239,89593.0,7789,9250,91054.0


In [0]:
summary_df.select(
    F.min("date_id").alias("min_date"),
    F.max("date_id").alias("max_date")
).show()

+--------+--------+
|min_date|max_date|
+--------+--------+
|20240101|20250831|
+--------+--------+



In [0]:
target_table = f"{catalog_name}.gold.{table_name}"

# First run

if not spark.catalog.tableExists(target_table):

    (
        summary_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

    spark.sql(
        f"""
        ALTER TABLE {target_table}
        CLUSTER BY AUTO
        """
    )

# Incremental updates

else:

    delta_table = DeltaTable.forName(
        spark,
        target_table
    )

    (
        delta_table.alias("gold_table")
        .merge(
            summary_df.alias("data_snapshot"),
            """
            gold_table.date_id = data_snapshot.date_id
            AND
            gold_table.currency = data_snapshot.currency
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )